In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
SRC  = ROOT / "src"
DATA = ROOT / "data"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from companies import HOLDINGS_INFO, RUSSELL_SECTOR_WEIGHTS
from portfolio_optimizer import (
    OptimizerBands,
    optimize_quadratic_portfolio,
    compute_alpha_vector,
    _build_benchmark_weights,
    _build_sector_exposure_matrix,
)
import pandas as pd
import numpy as np

In [2]:
assert isinstance(HOLDINGS_INFO, dict) and len(HOLDINGS_INFO) > 0
assert isinstance(RUSSELL_SECTOR_WEIGHTS, dict) and len(RUSSELL_SECTOR_WEIGHTS) > 0

tickers = sorted(HOLDINGS_INFO.keys())
print("Universe size:", len(tickers))
print("First 10:", tickers[:10])

# Ensure all have asset_type + sector
missing_keys = [t for t,m in HOLDINGS_INFO.items() if "asset_type" not in m or "sector" not in m]
print("Tickers missing asset_type/sector:", missing_keys)


Universe size: 26
First 10: ['ACHR', 'AGCO', 'AUR', 'CALY', 'CVLT', 'EHC', 'ELLO', 'ENSG', 'INMD', 'LASR']
Tickers missing asset_type/sector: []


In [3]:
alpha = compute_alpha_vector(tickers=tickers, holdings_info=HOLDINGS_INFO)

print("Alpha min/max:", float(alpha.min()), float(alpha.max()))
print("Alpha finite:", np.isfinite(alpha).all())

n_etf = sum(HOLDINGS_INFO[t].get("asset_type") == "index" for t in tickers)
n_stk = len(tickers) - n_etf
print("Num ETFs:", n_etf, "Num stocks:", n_stk)

assert len(alpha) == len(tickers)
assert np.isfinite(alpha).all()


Alpha min/max: -0.0055 0.24999999999999997
Alpha finite: True
Num ETFs: 9 Num stocks: 17


In [4]:
w_b, sectors_used, b_sec = _build_benchmark_weights(
    tickers=tickers,
    holdings_info=HOLDINGS_INFO,
    russell_sector_weights=RUSSELL_SECTOR_WEIGHTS,
)

print("w_b sum:", float(w_b.sum()), "min/max:", float(w_b.min()), float(w_b.max()))
print("sectors_used:", len(sectors_used), sectors_used)

assert len(w_b) == len(tickers)
assert abs(float(w_b.sum()) - 1.0) < 1e-6
assert np.isfinite(w_b).all()


w_b sum: 1.0 min/max: 0.0 0.19553655229950986
sectors_used: 9 ['Health Care', 'Industrials', 'Financials', 'Information Technology', 'Consumer Discretionary', 'Real Estate', 'Energy', 'Materials', 'Utilities']


In [5]:
S = _build_sector_exposure_matrix(
    tickers=tickers,
    holdings_info=HOLDINGS_INFO,
    sectors_used=sectors_used,
)
print("S shape:", S.shape)

assert S.shape == (len(sectors_used), len(tickers))
assert np.isfinite(S).all()

# check that each column is either all zeros (unknown sector) or exactly 1 in one row
col_sums = S.sum(axis=0)
print("Column sums unique:", sorted(set(col_sums.tolist()))[:10])


S shape: (9, 26)
Column sums unique: [1.0]


In [6]:
prices = pd.read_csv(DATA/"daily_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
missing_cols = [t for t in tickers if t not in prices.columns]
print("Missing price columns:", missing_cols)
assert not missing_cols, f"daily_prices.csv missing: {missing_cols}"

px = prices[tickers].ffill()
rets = px.pct_change().dropna()
print("Returns shape:", rets.shape)
print("Any NaNs in returns:", rets.isna().any().any())
assert not rets.isna().any().any()


Missing price columns: []
Returns shape: (276, 26)
Any NaNs in returns: False


In [12]:
bands = OptimizerBands(
    etf_min=0.20, etf_max=0.35,
    stock_min=0.65, stock_max=0.8,
    max_weight_etf=0.09,
    max_weight_stock=0.07,
    sector_penalty_gamma=0.2,  # turn off penalty to isolate TE first
)

res = optimize_quadratic_portfolio(
    daily_prices_csv=DATA/"daily_prices.csv",
    holdings_info=HOLDINGS_INFO,
    russell_sector_weights=RUSSELL_SECTOR_WEIGHTS,
    te_cap=0.15,        # huge, should be feasible if TE modeling correct
    bands=bands,
    lookback_days=252,
    debug=True,
    mosek_log=False,    # set True if you want full MOSEK interior-point logs
)

print("Result is None?", res is None)
if res is not None:
    w = pd.Series(res["weights"]).sort_values(ascending=False)
    print("Solved TE:", res["te"], "IR:", res["ir"], "ActiveRet:", res["active_return"])
    display(w.head(15))


[OPT-DBG] Sigma shape=(26, 26), eig_min=1.370e-01, eig_max=2.808e+00, finite=True
[OPT-DBG] Alpha min=-0.005500, max=0.250000, finite=True
[OPT-DBG] w_b sum=1.000000, min=0.000000, max=0.195537, len=26
[OPT-DBG] Representable sectors used=9; missing Russell sectors ignored automatically.
[OPT-DBG] Universe n=26, ETFs=9, Stocks=17
[OPT-DBG] UB sum=2.000000, UB min=0.070000, UB max=0.090000
[OPT-DBG] ETF UB sum=0.810000, Stock UB sum=1.190000
[OPT-DBG] Sleeves: etf_min=0.2, etf_max=0.35, stock_min=0.65, stock_max=0.8
[OPT-DBG] Stage A: bounds+budget only (should ALWAYS be feasible if ub.sum>=1)
[OPT-DBG] Stage A ok. sum=1.000000, min=0.000000, max=0.070000
[OPT-DBG] Stage B: add sleeve constraints
[OPT-DBG] Stage B ok. ETF=0.2000, STOCK=0.8000
[OPT-DBG] Stage C: add TE constraint (this is where your issue most likely is)
[OPT-DBG] Cholesky OK (Sigma2 PSD).
[OPT-DBG] Stage C ok. realized TE=0.135884 cap=0.15
[OPT-DBG] Stage D: add sector penalty cone (should remain feasible)
[OPT-DBG] Cho

RSPF    0.090000
PSCI    0.090000
EHC     0.070000
PRVA    0.070000
ENSG    0.070000
STEP    0.070000
NXT     0.070000
UAA     0.070000
PSCH    0.068443
ACHR    0.056172
ELLO    0.049936
LASR    0.035967
PSCC    0.034551
CVLT    0.027653
SKY     0.022669
dtype: float64

In [14]:
bands2 = OptimizerBands(
    etf_min=0.20, etf_max=0.35,
    stock_min=0.65, stock_max=0.8,
    max_weight_etf=0.09,
    max_weight_stock=0.07,
    sector_penalty_gamma=0.2,  # turn off penalty to isolate TE first
)

res2 = optimize_quadratic_portfolio(
    daily_prices_csv=DATA/"daily_prices.csv",
    holdings_info=HOLDINGS_INFO,
    russell_sector_weights=RUSSELL_SECTOR_WEIGHTS,
    te_cap=0.14,      # a real cap
    bands=bands2,
    lookback_days=252,
    debug=True,
)

print("Result is None?", res2 is None)
if res2 is not None:
    w2 = pd.Series(res2["weights"]).sort_values(ascending=False)
    print("Solved TE:", res2["te"], "IR:", res2["ir"], "ActiveRet:", res2["active_return"])
    display(w2.head(15))
    print("Top sector deviations:")
    dev = pd.Series(res2["sector_deviation"]).sort_values(key=lambda s: s.abs(), ascending=False)
    display(dev.head(10))


[OPT-DBG] Sigma shape=(26, 26), eig_min=1.370e-01, eig_max=2.808e+00, finite=True
[OPT-DBG] Alpha min=-0.005500, max=0.250000, finite=True
[OPT-DBG] w_b sum=1.000000, min=0.000000, max=0.195537, len=26
[OPT-DBG] Representable sectors used=9; missing Russell sectors ignored automatically.
[OPT-DBG] Universe n=26, ETFs=9, Stocks=17
[OPT-DBG] UB sum=2.000000, UB min=0.070000, UB max=0.090000
[OPT-DBG] ETF UB sum=0.810000, Stock UB sum=1.190000
[OPT-DBG] Sleeves: etf_min=0.2, etf_max=0.35, stock_min=0.65, stock_max=0.8
[OPT-DBG] Stage A: bounds+budget only (should ALWAYS be feasible if ub.sum>=1)
[OPT-DBG] Stage A ok. sum=1.000000, min=0.000000, max=0.070000
[OPT-DBG] Stage B: add sleeve constraints
[OPT-DBG] Stage B ok. ETF=0.2000, STOCK=0.8000
[OPT-DBG] Stage C: add TE constraint (this is where your issue most likely is)
[OPT-DBG] Cholesky OK (Sigma2 PSD).
[OPT-DBG] Stage C ok. realized TE=0.128589 cap=0.14
[OPT-DBG] Stage D: add sector penalty cone (should remain feasible)
[OPT-DBG] Cho

RSPF    0.090000
PSCI    0.090000
PSCH    0.080060
EHC     0.070000
PRVA    0.070000
ENSG    0.070000
STEP    0.070000
NXT     0.070000
UAA     0.068926
ACHR    0.049481
ELLO    0.043630
PSCC    0.042109
LASR    0.031116
PSCT    0.026156
CVLT    0.023218
dtype: float64

Top sector deviations:


Health Care               0.110692
Energy                   -0.047782
Real Estate              -0.047032
Materials                -0.042027
Information Technology    0.039497
Industrials              -0.030828
Consumer Discretionary    0.029097
Financials               -0.019685
Utilities                 0.008068
dtype: float64